# DrugCLIP screening with Fpocket — no known binding site

The hardest case: a predicted structure and **no relative with a bound ligand**
to borrow a pocket from. Here you have to *detect* the site rather than
transplant it.

This is the weakest of the three routes, and the screen-pipeline README says so:

> Tools like Fpocket can be used, but **only half of those pockets are precise
> enough for DrugCLIP**

The paper measures the cost — EF1% drops from 29.3 (pocket aligned from a holo
structure) to **19.0** (Fpocket). Use `run_screen_alphafold.ipynb` instead
whenever any relative with a bound ligand exists.

### The problem this notebook addresses

Fpocket returns dozens of candidate cavities (32 for PGK2), ranked by its own
scoring. If the top one is only about half likely to be the real site, taking it
alone is a coin flip. Taking several improves the odds — but combining them
naively does not work.

The repo's `retrieve_mols()` max-pools raw scores:

```python
res = pocket_reps @ mol_reps.T
res = res.max(axis=0)          # no normalisation
```

The paper is explicit that this is wrong:

> because different pockets may have varying score ranges, this can introduce
> bias when applying max pooling. To address this, we normalize the scores using
> an **adjusted robust z-score** before performing the max pooling

$$z_i = \frac{x_i - \mathrm{median}(X)}{1.48 \cdot \mathrm{MAD}(X)}$$

Without it, whichever cavity happens to produce larger raw similarities
dominates every molecule's score, whether or not it is the real site — a large
greasy cavity would swamp a genuine pocket.

So this notebook scores each pocket **separately**, normalises per pocket, then
max-pools. It reports the per-pocket rankings too, because disagreement between
pockets is itself informative.

> **This is the 6-fold ensemble version.** The single-model notebook is
> `run_screen_fpocket.ipynb`; everything else about the method is identical.
>
> The paper uses an ensemble for anything it acts on:
>
> > we obtain six model weights through six-fold cross-validation. During virtual
> > screening, these six model weights are used to generate six different
> > predictions, which are then combined using **mean pooling**
>
> The same six models were used for NET, TRIP12 and the genome-wide screen, so
> they are general-purpose rather than tuned to any target.
>
> Run `fetch_weights.ipynb` first if `model/6_folds/` does not exist.

## Configuration

In [ ]:
from pathlib import Path

REPO     = Path("/root/GitRepos/DrugCLIP")
PIPELINE = Path("/root/GitRepos/DrugCLIP_screen_pipeline")
WORK     = REPO / "data" / "pgk2_fpocket"      # gitignored

UNIPROT  = "P07205"       # human PGK2
N_POCKETS = 5             # how many of Fpocket's ranked pockets to screen
RANK_BY  = "druggability" # "druggability" or "score"

MOL_CSV  = REPO / "data" / "pgk2" / "pgk2_ligands.csv"
N_CONF   = 10
WEIGHTS  = REPO / "model" / "dude_identity_90.pt"
FPOCKET  = Path("/root/fpocket/bin/fpocket")

EMB_DIR  = WORK / "emb"
WORK.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)
print("work dir:", WORK)
print("fpocket: ", "found" if FPOCKET.exists() else "MISSING - see SETUP.md")
print("weights: ", "found" if WEIGHTS.exists() else "MISSING")

# --- output identity -------------------------------------------------------
# METHOD names the pocket strategy; results are written as
#   data/results/<molecule-csv-stem>__<METHOD>.csv
# so the same test set can be run through all three notebooks and compared.
METHOD      = "alphafold_fpocket_6fold"   # AlphaFold model, pocket detected by Fpocket
RESULTS_DIR = REPO / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / (Path(MOL_CSV).stem + "__" + METHOD + ".csv")
print("results ->", RESULTS_CSV)

# --- ensemble --------------------------------------------------------------
# One checkpoint per cross-validation fold. Scores are averaged across folds.
FOLD_DIR = REPO / "model" / "6_folds"
FOLDS    = sorted(FOLD_DIR.glob("fold_*.pt"))
assert FOLDS, f"no checkpoints in {FOLD_DIR} - run fetch_weights.ipynb first"
print(f"{len(FOLDS)} folds:", ", ".join(f.stem for f in FOLDS))

In [ ]:
import sys, json, re, pickle, shutil, subprocess, urllib.request
import numpy as np, pandas as pd, lmdb, torch
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog("rdApp.*")
sys.path.insert(0, str(REPO))

def read_lmdb(path):
    env = lmdb.open(str(path), subdir=False, readonly=True, lock=False,
                    readahead=False, meminit=False, max_readers=256)
    with env.begin() as txn:
        out = [pickle.loads(v) for _, v in txn.cursor()]
    env.close()
    return out

print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())

## Step 1 — Structure, then Fpocket

The AlphaFold model is a download, not a computation — the database already
holds predictions for essentially every human protein.

In [ ]:
api = "https://alphafold.ebi.ac.uk/api/prediction/" + UNIPROT
meta = json.loads(urllib.request.urlopen(api).read())[0]
af_pdb = WORK / "AF_model.pdb"
if not af_pdb.exists():
    urllib.request.urlretrieve(meta["pdbUrl"], af_pdb)
print("gene", meta.get("gene"), "| version", meta.get("latestVersion"))

b = np.array([float(l[60:66]) for l in open(af_pdb) if l.startswith("ATOM")])
print("plDDT median %.1f | above 90: %.0f%% | below 70: %.1f%%"
      % (np.median(b), 100*(b > 90).mean(), 100*(b < 70).mean()))

In [ ]:
out_dir = WORK / "AF_model_out"
if not out_dir.exists():
    r = subprocess.run([str(FPOCKET), "-f", "AF_model.pdb"],
                       cwd=str(WORK), capture_output=True, text=True)
    print(r.stdout.strip()[-200:] or r.stderr[-400:])

pqrs = sorted((out_dir / "pockets").glob("pocket*_vert.pqr"))
print("Fpocket found", len(pqrs), "candidate cavities")

## Step 2 — Rank and select

Fpocket scores every cavity. **Druggability Score** estimates how likely a
cavity is to bind a drug-like molecule, and is usually the more relevant of the
two for screening; **Pocket Score** is its general-purpose ranking.

Neither is reliable enough to trust blindly — that is the whole reason for
taking several.

In [ ]:
info_txt = (out_dir / "AF_model_info.txt").read_text()
blocks = re.split(r"Pocket (\d+) :", info_txt)[1:]

rows = []
for num, body in zip(blocks[0::2], blocks[1::2]):
    def grab(key):
        m = re.search(re.escape(key) + r"\s*:\s*([-\d.]+)", body)
        return float(m.group(1)) if m else np.nan
    rows.append({"pocket": int(num),
                 "score": grab("Score"),
                 "druggability": grab("Druggability Score"),
                 "volume": grab("Volume"),
                 "alpha_spheres": grab("Number of Alpha Spheres")})

pk = pd.DataFrame(rows).sort_values(RANK_BY, ascending=False).reset_index(drop=True)
selected = pk.head(N_POCKETS)
display(pk.head(10).style.set_caption("Fpocket cavities, ranked by " + RANK_BY))
print("selected pockets:", list(selected["pocket"]))

## Step 3 — Turn each cavity into a pocket

Fpocket outputs *alpha-sphere centres* filling a cavity, not a ligand. The
pipeline's trick is to write those centres out as a fake ligand residue named
`STP`, so the ordinary 6 A rule has something to measure from:

```python
lig_res = Residue.Residue(('H_STP', 1, ' '), 'STP', ' ')   # build_fpocket_complex.py
```

Nothing downstream can tell the difference between this and a real bound ligand.

In [ ]:
sys.path.insert(0, str(PIPELINE))
from build_fpocket_complex import pqr_parser, build_complex

pdb_dir = WORK / "pdb_in"
if pdb_dir.exists():
    shutil.rmtree(pdb_dir)
pdb_dir.mkdir()

made = []
for p in selected["pocket"]:
    pqr = out_dir / "pockets" / ("pocket%d_vert.pqr" % p)
    stub = pdb_dir / ("p%02d.pdb" % p)
    shutil.copy(af_pdb, stub)
    build_complex(str(stub), pqr_parser(str(pqr)))   # writes p##_STP.pdb
    stub.unlink()                                     # keep only the _STP file
    made.append(pdb_dir / ("p%02d_STP.pdb" % p))
    print("pocket %2d -> %s  (%d alpha spheres)"
          % (p, made[-1].name, len(pqr_parser(str(pqr)))))

In [ ]:
# one LMDB per pocket, so each can be scored on its own
pocket_lmdbs = {}
for p, f in zip(selected["pocket"], made):
    one = WORK / ("pocket_%02d_dir" % p)
    if one.exists():
        shutil.rmtree(one)
    one.mkdir()
    shutil.copy(f, one / f.name)
    r = subprocess.run([sys.executable, str(PIPELINE / "pocket_from_pdb.py"),
                        str(one), "--name", "pocket"],
                       capture_output=True, text=True, cwd=str(PIPELINE))
    lm = one / "pocket.lmdb"
    if not lm.exists():
        print("pocket %2d FAILED - pocket_from_pdb.py swallows errors silently" % p)
        continue
    recs = read_lmdb(lm)
    if not recs:
        print("pocket %2d produced an EMPTY lmdb" % p)
        continue
    pocket_lmdbs[p] = lm
    print("pocket %2d -> %3d atoms  elements=%s"
          % (p, len(recs[0]["pocket_atoms"]), sorted(set(recs[0]["pocket_atoms"]))))

assert pocket_lmdbs, "no usable pockets"

## Step 4 — Molecule library

Unchanged from the other notebooks.

In [ ]:
df = pd.read_csv(MOL_CSV)

def to_record(name, smi, n_conf=N_CONF):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None, "unparseable SMILES"
    mol = Chem.AddHs(mol)
    if len(AllChem.EmbedMultipleConfs(mol, numConfs=n_conf, randomSeed=42,
                                      pruneRmsThresh=1)) == 0:
        return None, "no conformer embedded"
    try:
        AllChem.MMFFOptimizeMoleculeConfs(mol)
    except Exception:
        pass
    mol = Chem.RemoveHs(mol)
    return {"atoms": [a.GetSymbol() for a in mol.GetAtoms()],
            "coordinates": [np.array(mol.GetConformer(c).GetPositions(),
                                     dtype=np.float32)
                            for c in range(mol.GetNumConformers())],
            "smi": smi, "name": name}, None

records, failures = [], []
for _, row in df.iterrows():
    rec, err = to_record(row["name"], row["smiles"])
    (records.append(rec) if rec else failures.append((row["name"], err)))
print("built", len(records), "| failed", len(failures))

vocab = set((REPO / "data" / "dict_mol.txt").read_text().split())
unknown = set(a for r in records for a in r["atoms"]) - vocab
print("elements not in dict_mol.txt:", sorted(unknown) or "none")

mols_lmdb = WORK / "mols.lmdb"
if mols_lmdb.exists():
    mols_lmdb.unlink()
env = lmdb.open(str(mols_lmdb), subdir=False, readonly=False, lock=False,
                readahead=False, meminit=False, map_size=1099511627776)
with env.begin(write=True) as txn:
    for i, r in enumerate(records):
        txn.put(str(i).encode("ascii"), pickle.dumps(r))
env.close()
print(mols_lmdb.name, ":", len(read_lmdb(mols_lmdb)), "records")

## Screen every fold against every pocket

Two things are being combined here, and they solve different problems:

| | why |
| --- | --- |
| **z-score per pocket**, then max-pool | pockets have different score ranges; without this the loudest cavity dominates |
| **mean across folds** | reduces variance between models |

The paper specifies each separately but not how they interact. The order used
here is: normalise and max-pool across pockets **within** each fold, then average
those per-fold results. That keeps each step operating on comparable numbers.

Each fold again gets its own embedding cache, for the reason described in the
single-pocket notebooks.

In [ ]:
def screen_with(weights, emb_subdir, pocket_lmdb):
    emb = EMB_DIR / emb_subdir
    emb.mkdir(parents=True, exist_ok=True)
    out = emb / "ranked_compounds.txt"
    if out.exists():
        out.unlink()

    cmd = [sys.executable, "./unimol/retrieval.py",
           "--user-dir", "./unimol", str(REPO / "data"),
           "--valid-subset", "test", "--results-path", str(WORK / "results"),
           "--num-workers", "0", "--ddp-backend", "c10d", "--batch-size", "8",
           "--task", "drugclip", "--loss", "in_batch_softmax", "--arch", "drugclip",
           "--max-pocket-atoms", "256", "--seed", "1",
           "--path", str(weights),
           "--log-interval", "100", "--log-format", "simple",
           "--mol-path", str(mols_lmdb), "--pocket-path", str(pocket_lmdb),
           "--emb-dir", str(emb)]
    if not torch.cuda.is_available():
        cmd.append("--cpu")
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO))
    if not out.exists():
        raise RuntimeError(f"failed:\n{r.stderr[-1500:]}")
    d = {}
    for line in out.read_text().splitlines():
        if line.strip():
            smi, s = line.rsplit("\t", 1)
            d[smi] = float(s)
    return d


# scores[fold][pocket] = {smiles: score}
scores = {}
for f in FOLDS:
    scores[f.stem] = {}
    for p, lm in pocket_lmdbs.items():
        scores[f.stem][p] = screen_with(f, f"{f.stem}_p{p:02d}", lm)
    print(f"{f.stem}: {len(pocket_lmdbs)} pockets done")

## Combine, then report

Per fold: z-score each pocket, take the best pocket for each molecule. Then
average those across folds.

In [ ]:
def robust_z(x):
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return np.zeros_like(x) if mad == 0 else (x - med) / (1.48 * mad)

smis  = [r["smi"] for r in records]
names = dict((r["smi"], r["name"]) for r in records)

fold_z, fold_raw, fold_best = {}, {}, {}
for fold, by_pocket in scores.items():
    raw = pd.DataFrame({p: [by_pocket[p][s] for s in smis] for p in by_pocket},
                       index=smis)
    z = raw.apply(robust_z, axis=0)        # normalise each pocket first
    fold_z[fold]    = z.max(axis=1)        # then best pocket per molecule
    fold_raw[fold]  = raw.max(axis=1)
    fold_best[fold] = z.idxmax(axis=1)

Z   = pd.DataFrame(fold_z)
RAW = pd.DataFrame(fold_raw)

res = pd.DataFrame({
    "name": [names[s] for s in smis],
    "smiles": smis,
    "score_raw": RAW.mean(axis=1).values,
    "score_std": Z.std(axis=1).values,
    "zscore": Z.mean(axis=1).values,       # mean across folds
    "best_pocket": pd.DataFrame(fold_best).mode(axis=1)[0].values,
})
res = res.sort_values("zscore", ascending=False).reset_index(drop=True)
res["rank"] = np.arange(1, len(res) + 1)
res["method"] = METHOD
res = res[["rank", "name", "smiles", "score_raw", "score_std",
           "zscore", "method", "best_pocket"]]
res.to_csv(RESULTS_CSV, index=False)

print("wrote", RESULTS_CSV, f"({len(res)} molecules)")
display(res.head(20))

zz = Z.copy()
zz.index = [names[s] for s in smis]
print("\nper-fold ensemble z-score:")
display(zz.round(2))
print("`best_pocket` is the pocket most often chosen across folds.")